# Oracle AI Database Vector Search
This notebook implements the Vector Search portion of the lab, completely skipping the OCI GenAI requirement. It uses the Oracle DB wallet connection details you provided.

In [1]:
import os
import shutil
import oracledb
import pandas as pd
import json

# Connection Configurations
username = "ADMIN"
password = "ppPPPP__253fSEDF8675__3fcdvbj"
dsn = "adbforailowercost_high"
wallet_zip = "/home/opc/vector-playground/Wallet_ADBforAIlowerCost_22-Feb-2026.zip"
wallet_dir = "/home/opc/vector-playground/wallet"

# Unzip wallet if not already done
if not os.path.exists(wallet_dir):
    print(f"Unzipping wallet to {wallet_dir}...")
    import zipfile
    with zipfile.ZipFile(wallet_zip, 'r') as zip_ref:
        zip_ref.extractall(wallet_dir)
    print("✅ Wallet unzipped.")
else:
    print(f"✅ Wallet directory already exists at {wallet_dir}")

print("Connecting to Oracle Database...")
try:
    # Thin mode connection with wallet
    connection = oracledb.connect(
        user=username,
        password=password,
        dsn=dsn,
        config_dir=wallet_dir,
        wallet_location=wallet_dir,
        wallet_password=password
    )
    print("✅ Connection successful!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

cursor = connection.cursor()

✅ Wallet directory already exists at /home/opc/vector-playground/wallet
Connecting to Oracle Database...
✅ Connection successful!


## 1. Fetch Customer Data
Queries the `customers_dv` duality view to verify data connectivity and fetch customer `100001`.

In [2]:
def fetch_customer_data(customer_id):
    cursor.execute(
        "SELECT data FROM customers_dv WHERE JSON_VALUE(data, '$._id') = :customer_id",
        {'customer_id': customer_id}
    )
    result = cursor.fetchone()
    if not result: return None
    # Handle CLOB read
    val = result[0].read() if hasattr(result[0], 'read') else result[0]
    return json.loads(val) if isinstance(val, str) else val

selected_customer_id = 100001
customer_json = fetch_customer_data(selected_customer_id)

if customer_json:
    print(f"Found Customer Info: {customer_json.get('FirstName', '')} {customer_json.get('LastName', '')}")
else:
    print("Warning: Customer 100001 not found. Did you run the 'setup_db.py' script from the lab?")

Found Customer Info: Dan Thompson


## 2. Prepare Mock Recommendation Text
Since we are skipping OCI GenAI, we're providing a localized dummy response to chunk, store, and vectorize.

In [3]:
mock_recommendation = """
1. Comprehensive Evaluation
James Smith has a solid credit score but quite a bit of existing debt. The combination of income and credit history makes James a reasonable candidate for several loan types.

2. Top 3 Loan Recommendations
- Loan A: 30-Year Fixed Mortgage at 6.5% interest. Ideal for stable, long-term payments.
- Loan B: 15-Year Fixed Mortgage at 5.8% interest. Better interest rate but higher monthly payments.
- Loan C: FHA Loan at 6.0% interest. Good for first-time buyers with lower down payment requirements.

3. Recommendations Explanations
The 30-year fixed is recommended because it keeps the monthly payment affordable given the existing debt-to-income ratio. The 15-year is presented as a faster payoff alternative.

4. Final Suggestion
We suggest proceeding with Loan A (30-Year Fixed) so that James has a comfortable emergency buffer in his monthly budget.
"""
print("✅ Mock recommendation prepared.")

✅ Mock recommendation prepared.


## 3. Chunk and Store the Document
We chunk the text into multiple smaller blocks using Oracle's `VECTOR_CHUNKS` function, and store it.

In [4]:
# Clean any prior chunks for this customer
cursor.execute("DELETE FROM LOAN_CHUNK WHERE CUSTOMER_ID = :cust_id", {'cust_id': selected_customer_id})
connection.commit()

chunk_sizes = [50]

# Insert chunks
for size in chunk_sizes:
    insert_sql = f"""
        INSERT INTO LOAN_CHUNK (CUSTOMER_ID, CHUNK_ID, CHUNK_TEXT)
        SELECT :cust_id,
            :chunk_size + vc.chunk_offset,
            vc.chunk_text
        FROM (SELECT :rec_text AS txt FROM dual) s,
            VECTOR_CHUNKS(
            dbms_vector_chain.utl_to_text(s.txt)
            BY words
            MAX {size}
            OVERLAP 0
            SPLIT BY sentence
            LANGUAGE american
            NORMALIZE all
            ) vc
    """
    cursor.execute(
        insert_sql,
        {'cust_id': selected_customer_id, 'chunk_size': size, 'rec_text': mock_recommendation}
    )

# Fetch chunks to verify
cursor.execute("""
SELECT CHUNK_ID, CHUNK_TEXT
  FROM LOAN_CHUNK
 WHERE CUSTOMER_ID = :cust_id
  ORDER BY CHUNK_ID
""", {'cust_id': selected_customer_id})
rows = cursor.fetchall()

items = []
for cid, ctext in rows:
    txt = ctext.read() if hasattr(ctext, 'read') else ctext
    txt = txt or ""
    items.append({
        "CHUNK_ID": cid,
        "Chars": len(txt),
        "Preview": (txt[:100] + "...") if len(txt) > 100 else txt
    })

df_chunks = pd.DataFrame(items)
connection.commit()
print("✅ Chunking complete!")
display(df_chunks)

✅ Chunking complete!


,CHUNK_ID,Chars,Preview
0,52,205,1. Comprehensive Evaluation James Smith has a ...
1,259,162,Top 3 Loan Recommendations-Loan A: 30-Year Fix...
2,426,152,Better interest rate but higher monthly paymen...
3,582,210,Recommendations Explanations The 30-year fixed...
4,794,138,Final Suggestion We suggest proceeding with Lo...


## 4. DB Data Embedding
Since the `DEMO_MODEL` ONNX model is missing from the database, we'll generate the vector embeddings locally in Python using `sentence-transformers`, and upload those vector arrays directly into the Oracle database `VECTOR` column.

In [5]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    import os
    os.system("python3 -m pip install sentence-transformers")
    from sentence_transformers import SentenceTransformer

print("Loading lightweight embedding model locally...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Fetching chunks to embed...")
cursor.execute("SELECT CHUNK_ID, CHUNK_TEXT FROM LOAN_CHUNK WHERE CUSTOMER_ID = :cust_id", {'cust_id': selected_customer_id})
chunks = cursor.fetchall()

print(f"Generating embeddings for {len(chunks)} chunks...")
for cid, ctext in chunks:
    text = ctext.read() if hasattr(ctext, 'read') else ctext
    # Generate 384-dimensional vector and convert to string for Oracle
    embedding = model.encode(text).tolist()
    vec_str = str(embedding)
    
    # Update the row with the new vector array
    cursor.execute("""
        UPDATE LOAN_CHUNK
           SET CHUNK_VECTOR = TO_VECTOR(:vec)
         WHERE CUSTOMER_ID = :cust_id AND CHUNK_ID = :cid
    """, {'cust_id': selected_customer_id, 'cid': cid, 'vec': vec_str})
    
connection.commit()
print("✅ Local vectorization and DB update complete.")

/home/opc/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading lightweight embedding model locally...
Fetching chunks to embed...
Generating embeddings for 5 chunks...
✅ Local vectorization and DB update complete.


## 5. Vector Search against the Embeddings
We use the same local model to vectorize our search question, and query the DB using `.VECTOR_DISTANCE()`.

In [6]:
question = "Which loan do you suggest for a faster payoff and what is the interest rate?"
print(f"Searching for: '{question}'\n")

# Vectorize the query locally and convert to string for Oracle
q_vec = model.encode(question).tolist()
q_vec_str = str(q_vec)

# Perform similarity search in database 
cursor.execute("""
    SELECT CHUNK_ID, CHUNK_TEXT, VECTOR_DISTANCE(CHUNK_VECTOR, TO_VECTOR(:qv), COSINE) as dist
    FROM LOAN_CHUNK
    WHERE CUSTOMER_ID = :cust_id
    AND CHUNK_VECTOR IS NOT NULL
    ORDER BY dist ASC
    FETCH FIRST 3 ROWS ONLY
""", {'cust_id': selected_customer_id, 'qv': q_vec_str})

retrieved = cursor.fetchall()

for rank, row in enumerate(retrieved):
    cid = row[0]
    text = row[1].read() if hasattr(row[1], 'read') else row[1]
    dist = row[2]
    print(f"⭐ Rank {rank+1} [Distance: {dist:.4f}] - Chunk {cid}")
    text_clean = text.strip().replace('\n', ' ')
    print(f"   Text: {text_clean}\n")


Searching for: 'Which loan do you suggest for a faster payoff and what is the interest rate?'

⭐ Rank 1 [Distance: 0.3229] - Chunk 259
   Text: Top 3 Loan Recommendations-Loan A: 30-Year Fixed Mortgage at 6.5% interest. Ideal for stable, long-term payments.-Loan B: 15-Year Fixed Mortgage at 5.8% interest.

⭐ Rank 2 [Distance: 0.3506] - Chunk 426
   Text: Better interest rate but higher monthly payments.-Loan C: FHA Loan at 6.0% interest. Good for first-time buyers with lower down payment requirements. 3.

⭐ Rank 3 [Distance: 0.5127] - Chunk 582
   Text: Recommendations Explanations The 30-year fixed is recommended because it keeps the monthly payment affordable given the existing debt-to-income ratio. The 15-year is presented as a faster payoff alternative. 4.



## 6. Extract Thai PDF to Database
We are adding processing capability for Thai documents. We extract `2.3.6.1 ทรัพย์สินประเภทที่ธนาคารรับเป็นหลักประกัน.pdf` using `PyMuPDF`.

In [7]:
try:
    import fitz
except ImportError:
    print("Installing PyMuPDF...")
    import os
    os.system("python3 -m pip install PyMuPDF")
    import fitz

pdf_path = "/home/opc/vector-playground/documents/2.3.6.1 ทรัพย์สินประเภทที่ธนาคารรับเป็นหลักประกัน.pdf"
print(f"Reading PDF: {pdf_path}")

doc = fitz.open(pdf_path)
pdf_text = ""
for page_num in range(len(doc)):
    pdf_text += doc[page_num].get_text("text") + "\n"

print(f"Extracted {len(pdf_text)} characters from Thai PDF.")

Reading PDF: /home/opc/vector-playground/documents/2.3.6.1 ทรัพย์สินประเภทที่ธนาคารรับเป็นหลักประกัน.pdf
Extracted 16836 characters from Thai PDF.


## 7. Chunk & Embed PDF Document
We clear out previous chunks for a different ID (e.g., PDF_999) and store the PDF text.

In [9]:
pdf_customer_id = "PDF_999" # Dummy ID for the PDF chunks
cursor.execute("DELETE FROM LOAN_CHUNK WHERE CUSTOMER_ID = :cust_id", {'cust_id': pdf_customer_id})
connection.commit()

# Chunk the PDF text
size = 500
insert_sql = f"""
    INSERT INTO LOAN_CHUNK (CUSTOMER_ID, CHUNK_ID, CHUNK_TEXT)
    SELECT :cust_id,
        :chunk_size + vc.chunk_offset,
        vc.chunk_text
    FROM (SELECT :rec_text AS txt FROM dual) s,
        VECTOR_CHUNKS(
        dbms_vector_chain.utl_to_text(s.txt)
        BY words
        MAX {size}
        OVERLAP 0
        SPLIT BY sentence
        LANGUAGE american
        NORMALIZE all
        ) vc
"""
cursor.setinputsizes(rec_text=oracledb.DB_TYPE_CLOB)
cursor.execute(insert_sql, {'cust_id': pdf_customer_id, 'chunk_size': size, 'rec_text': pdf_text})
connection.commit()

# Embed PDF Chunks
cursor.execute("SELECT CHUNK_ID, CHUNK_TEXT FROM LOAN_CHUNK WHERE CUSTOMER_ID = :cust_id", {'cust_id': pdf_customer_id})
pdf_chunks = cursor.fetchall()

print(f"Generating embeddings for {len(pdf_chunks)} PDF chunks...")
for cid, ctext in pdf_chunks:
    text = ctext.read() if hasattr(ctext, 'read') else ctext
    embedding = model.encode(text or "").tolist()
    vec_str = str(embedding)
    
    cursor.execute("""
        UPDATE LOAN_CHUNK
           SET CHUNK_VECTOR = TO_VECTOR(:vec)
         WHERE CUSTOMER_ID = :cust_id AND CHUNK_ID = :cid
    """, {'cust_id': pdf_customer_id, 'cid': cid, 'vec': vec_str})

connection.commit()
print("✅ PDF chunking and embedding complete.")

Generating embeddings for 33 PDF chunks...
✅ PDF chunking and embedding complete.


## 8. Querying the Thai PDF
Now we can try searching for specific Thai rules within the KBank document.

In [10]:
pdf_question = "ที่ดินเปล่าเพื่อการพาณิชย์และเพื่อที่อยู่อาศัยรับเป็นหลักประกันได้กี่เปอเซ็นต์?"
print(f"Searching for: '{pdf_question}'\n")

q_vec_pdf = model.encode(pdf_question).tolist()
q_vec_str_pdf = str(q_vec_pdf)

cursor.execute("""
    SELECT CHUNK_ID, CHUNK_TEXT, VECTOR_DISTANCE(CHUNK_VECTOR, TO_VECTOR(:qv), COSINE) as dist
    FROM LOAN_CHUNK
    WHERE CUSTOMER_ID = :cust_id
    AND CHUNK_VECTOR IS NOT NULL
    ORDER BY dist ASC
    FETCH FIRST 3 ROWS ONLY
""", {'cust_id': pdf_customer_id, 'qv': q_vec_str_pdf})

retrieved_pdf = cursor.fetchall()

for rank, row in enumerate(retrieved_pdf):
    cid = row[0]
    text = row[1].read() if hasattr(row[1], 'read') else row[1]
    dist = row[2]
    print(f"⭐ Rank {rank+1} [Distance: {dist:.4f}] - Chunk {cid}")
    text_clean = text.strip().replace('\n', ' ')
    print(f"   Text: {text_clean}\n")

Searching for: 'ที่ดินเปล่าเพื่อการพาณิชย์และเพื่อที่อยู่อาศัยรับเป็นหลักประกันได้กี่เปอเซ็นต์?'

⭐ Rank 1 [Distance: 0.6513] - Chunk 11386
   Text: ประเภททรัพย์ที่เป็นหลักทรัพย์-ในกรณีที่มีการรับหลักประกันประเภทหลักทรัพย์ตามที่ระบุในนโยบายเครดิตบทที่ 2.3.6.1 ทรัพย์สินประเภทที่ธนาคารรับเป็นหลักประกัน และ 2.3.6.3 ทรัพย์สินประเภทที่ธนาคารไม่รับเป็นหลักประกันไม่ให้มีการกำหนดปรับวงเงินสินเชื่อเพิ่มตามราคาหลักทรัพย์ที่มีการเปลี่ยนแปลงโดยอัติโนมัติ หากต้องการเพิ่มวงเงินภายใต้หลักทรัพย์เดิมต้องมีการลงนามในสัญญาสินเชื่อเพื่อเพิ่มวงเงินใหม่ทุกครั้ง-การให้สินเชื่อที่มีหลักทรัพย์เป็นหลักประกัน กรณีตรวจพบว่าลูกค้านำเงินสินเชื่อไปเพื่อเก็งกำไรในตลาดหลักทรัพย์ให

⭐ Rank 2 [Distance: 0.6585] - Chunk 4161
   Text: ทะลุ (อาคารเจาะทะลุถึงกัน) หมายเหตุ กรณีประเมินราคาก่อนวันที่ 1 มิ.ย. 63 กำหนดให้ LTV ไม่เกิน 80% ​95% ​22 ​ที่ดินพร้อมสิ่งปลูกสร้าง ที่มีเอกสารสิทธิ์ถูกต้องตามกฎหมาย* ซึ่งเอกสารสิทธิ์ออก ให้หลังวันประกาศของพื้นที่ที่ทับซ้อนกับเขตนิคมสร้างตนเอง,เขตปฏิรูปเพื่อ การเกษตร,จัดรูปเพื่อการเกษตร แต่